# Bills Summarizer


In [1]:
# check the device
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
from transformers import AutoTokenizer, LEDForConditionalGeneration

model_name = "Anurag33Gaikwad/legal-led-billsum-summarization"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = LEDForConditionalGeneration.from_pretrained(model_name).to(device)
model.eval()

text = """
THE DIGITAL PERSONAL DATA PROTECTION BILL, 2023

A Bill to provide for the processing of digital personal data
in a manner that recognises both the right of individuals to
protect their personal data and the need to process such
personal data for lawful purposes.

BE it enacted by Parliament as follows:

1. Short title and commencement.

The Bill shall come into force on such date as the Central
Government may, by notification, appoint.

2. Definitions.

In this Bill, unless the context otherwise requires, the
relevant terms relating to digital personal data, data
principals, data fiduciaries and processing shall have the
meanings assigned to them under the Bill.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=4096
)

# Move input tensors to GPU
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

global_attention_mask = torch.zeros_like(inputs["input_ids"], device=device)
global_attention_mask[:, 0] = 1

summary_ids = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    global_attention_mask=global_attention_mask,
    num_beams=5,
    max_length=512,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/296 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Input ids are automatically padded from 158 to 1024 to be a multiple of `config.attention_window`: 1024


This bill amends the Digital Personal Data Protection Act to provide for the processing of digital personal data in a manner that recognises both the right of individuals to self-protect their personal data and the need to process such personal data for lawful purposes.


In [3]:
print('Summary: ', summary)

Summary:  This bill amends the Digital Personal Data Protection Act to provide for the processing of digital personal data in a manner that recognises both the right of individuals to self-protect their personal data and the need to process such personal data for lawful purposes.


In [4]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


Evaluate Rouge Score for a single text

In [5]:
import evaluate

rouge = evaluate.load("rouge")

reference_summary = """
The Digital Personal Data Protection Bill, 2023 establishes a framework
for processing digital personal data while protecting individuals' data
rights and allowing lawful data processing.
"""

results = rouge.compute(
    predictions=[summary],
    references=[reference_summary]
)

print("Generated Summary:")
print(summary)

print("\nROUGE Scores:")
print("ROUGE-1:", results["rouge1"])
print("ROUGE-2:", results["rouge2"])
print("ROUGE-L:", results["rougeL"])

Generated Summary:
This bill amends the Digital Personal Data Protection Act to provide for the processing of digital personal data in a manner that recognises both the right of individuals to self-protect their personal data and the need to process such personal data for lawful purposes.

ROUGE Scores:
ROUGE-1: 0.48571428571428565
ROUGE-2: 0.1764705882352941
ROUGE-L: 0.4


Evaluate Rouge on Test Set

In [6]:
from datasets import load_dataset

# Load the test dataset
dataset = load_dataset('FiscalNote/billsum')
dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 1237
    })
})

In [7]:
test_dataset = dataset['test'].select(range(50))
test_dataset

Dataset({
    features: ['text', 'summary', 'title'],
    num_rows: 50
})

In [8]:
def compute_rouge(test_dataset):
    predictions = []
    references = []

    for i in range(len(test_dataset)):
        inputs = tokenizer(
            test_dataset[i]['text'],
            return_tensors="pt",
            truncation=True,
            max_length=4096
        )

        # Move input tensors to GPU
        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        global_attention_mask = torch.zeros_like(inputs["input_ids"], device=device)
        global_attention_mask[:, 0] = 1

        with torch.no_grad():
            summary_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                global_attention_mask=global_attention_mask,
                num_beams=5,
                max_length=512,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        predictions.append(summary)
        references.append(test_dataset[i]['summary'])

    results = rouge.compute(
        predictions=predictions,
        references=references
    )

    return results

In [9]:
rouge_score = compute_rouge(test_dataset)
print(rouge_score)

Input ids are automatically padded from 2140 to 3072 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 2588 to 3072 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 1957 to 2048 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 3633 to 4096 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 2204 to 3072 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 2756 to 3072 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 3314 to 4096 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 2087 to 3072 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 3227 to 4096 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 3612 to 4096 to

{'rouge1': np.float64(0.4496275980163997), 'rouge2': np.float64(0.28608830188641765), 'rougeL': np.float64(0.36580527543691094), 'rougeLsum': np.float64(0.3987544766274769)}


In [10]:
save_path = "./legal_led_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!
